# ITCHI v0.1 — caso Willa (2018): radios paramétricos W06 e índice diario

Objetivos:

1. Estimar R34 y el radio exterior con el perfil de Willoughby et al. (2006) en tres configuraciones y compararlos con los radios observados de IBTrACS y con el valor publicado por Pérez-Alarcón et al. (2021) para Willa.
2. Calcular ITCHI con MSWEP **diario** y percentiles **diarios** (opción B): el pipeline se evalúa en cada tiempo sinóptico con el campo del día y se toma el máximo intradiario celda a celda, lo que equivale a usar la unión de las máscaras de las posiciones del día.
3. Evaluar la sensibilidad a la elección del perfil de viento (Rankine de `wind.py` frente a W06) y del umbral exterior (2 m s⁻¹ frente a 5 kt).

Antes de correr: revisa la celda de parámetros (rutas de MSWEP, nomenclatura de archivos y convención temporal del archivo diario).

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from itchi.aggregation import compute_event_accumulated, compute_event_max
from itchi.climatology import (
    compute_standard_precipitation_climatology,
    extract_standard_percentiles,
)
from itchi.config import get_project_root, load_default_config
from itchi.constants import QUADRANTS
from itchi.geometry import build_geometry_fields
from itchi.ibtracs import (
    _decode_text_sequence,  # helper privado, solo para decodificar nombres
    get_ibtracs_storm_ids,
    ibtracs_to_track_dataframe,
    read_ibtracs_dataset,
)
from itchi.mswep import (
    extract_mswep_precipitation,
    infer_mswep_lat_lon_names,
    read_mswep_dataset,
    standardize_mswep_time,
    subset_mswep_domain,
)
from itchi.parametric_radii import (
    add_parametric_radii,
    compute_w06_wind_hazard,
    parametric_direct_fallback_km,
)
from itchi.pipeline import compute_itchi_snapshot_from_grid
from itchi.plotting import plot_itchi_field
from itchi.tracks import DEFAULT_R34_COLUMN_MAP, filter_synoptic_times
from itchi.units import NAUTICAL_MILE_TO_KM
from itchi.wind import compute_radial_wind_profile
from itchi.wind_profiles import KNOT_TO_MS, V_R5_MS, V_R34_MS, Willoughby2006Profile

## 0. Parámetros

In [ ]:
ROOT = get_project_root()
CFG = load_default_config()
PR = CFG.get("parametric_radii", {})

STORM_NAME, SEASON = "WILLA", 2018
T_CHECK = pd.Timestamp("2018-10-24 00:00")   # tiempo con radio publicado en PA21
R_OUTER_PA21_KM = 812.0

IBTRACS_FILE = ROOT / CFG["paths"]["tracks"] / "IBTrACS.EP.v04r01.nc"

# MSWEP diario (WSL). Verifica nomenclatura y convención temporal.
MSWEP_DAILY_DIR = Path("/mnt/d/MSWEP_V280/merged/Daily")
MSWEP_FILE_PATTERN = "{date:%Y%j}.nc"      # p. ej. 2018297.nc (año + día juliano)
MSWEP_DAILY_STAMP = "start"                # "start": la marca es el inicio del día acumulado
PRECIP_VAR = CFG["precipitation"]["variable_name"]

# Climatología diaria Q90/Q95/Q99 (mayo-noviembre, todos los días, incluidos secos)
CLIM_YEARS = (2000, 2024)
CLIM_MONTHS = tuple(range(5, 12))
PAD_DEG = 10.0                             # margen alrededor de la trayectoria
CLIM_FILE = ROOT / CFG["paths"]["climatology"] / f"mswep_daily_q_{STORM_NAME.lower()}{SEASON}.nc"

THRESHOLDS = {"r_outer_km": PR.get("outer_threshold_ms", 2.0), "r34_km": V_R34_MS}
RMW_UNIT = PR.get("rmw_unit", "km")        # tras la corrección de ibtracs.py: "km"
assert RMW_UNIT == "km", "rmw_km ya sale en km de ibtracs.py; evita doble conversión"

FIG_DIR = ROOT / CFG["outputs"]["figures"] / f"{STORM_NAME.lower()}{SEASON}"
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Trayectoria IBTrACS

In [ ]:
ds_ib = read_ibtracs_dataset(IBTRACS_FILE)
sids = np.asarray(get_ibtracs_storm_ids(ds_ib))
names = np.asarray(_decode_text_sequence(ds_ib["name"].values))
seasons = np.asarray(ds_ib["season"].values).astype(int)

match = sids[(names == STORM_NAME) & (seasons == SEASON)]
assert match.size == 1, f"coincidencias: {match}"
SID = str(match[0])

track = ibtracs_to_track_dataframe(ds_ib, SID)          # rmw_km en km
track = filter_synoptic_times(track).dropna(subset=["vmax_kt", "lat", "lon"])
track = track.reset_index(drop=True)
print(SID, len(track), "registros sinópticos")
track.head(12)

## 2. Radios paramétricos W06

Tres configuraciones:

- `pa21`: réplica de Pérez-Alarcón et al. (2021): Rmax y forma por regresión (coeficientes de su Ec. 6).
- `obs_rmw`: RMW observado (IBTrACS), relleno por regresión donde falte.
- `calib_r34`: RMW observado y X1 calibrado para que V(R34 medio observado) = 34 kt.

In [ ]:
CONFIGS = {
    "pa21": dict(use_observed_rmw=False, calibrate_to_r34=False, rmax_variant="pa21"),
    "obs_rmw": dict(use_observed_rmw=True, calibrate_to_r34=False),
    "calib_r34": dict(use_observed_rmw=True, calibrate_to_r34=True),
}

for name, kwargs in CONFIGS.items():
    track = add_parametric_radii(
        track, rmw_unit=RMW_UNIT, r34_unit="nm", thresholds_ms=THRESHOLDS,
        prefix=f"{name}_", **kwargs,
    )

# R34 medio observado (cuadrantes no nulos), km
r34_cols = [DEFAULT_R34_COLUMN_MAP[q] for q in QUADRANTS]
r34_nm = track[r34_cols].where(track[r34_cols] > 0)
track["r34_obs_km"] = r34_nm.mean(axis=1) * NAUTICAL_MILE_TO_KM

cols = ["time", "vmax_kt", "lat", "rmw_km", "r34_obs_km"] + [
    f"{c}_{v}" for c in CONFIGS for v in ("r34_km", "r_outer_km")
] + ["calib_r34_x1_calibrated"]
track[cols].round(1)

In [ ]:
# Verificación contra PA21 (Willa, 2018-10-24 00 UTC: 812 km)
row = track.loc[track["time"] == T_CHECK].squeeze()
r_pa21 = float(row["pa21_r_outer_km"])
print(f"W06 (pa21) = {r_pa21:.1f} km | PA21 = {R_OUTER_PA21_KM} km | "
      f"diferencia = {100 * (r_pa21 / R_OUTER_PA21_KM - 1):+.1f} %")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(track["time"], track["vmax_kt"], "k-o", ms=3)
axes[0].axhline(34, color="0.5", ls="--", lw=0.8)
axes[0].set_ylabel("Vmax (kt)")

axes[1].plot(track["time"], track["r34_obs_km"], "ko", ms=4, label="R34 obs (media)")
for name, style in zip(CONFIGS, ("-", "--", ":")):
    axes[1].plot(track["time"], track[f"{name}_r34_km"], "C0" + style, label=f"R34 {name}")
    axes[1].plot(track["time"], track[f"{name}_r_outer_km"], "C3" + style, label=f"R_out {name}")
axes[1].set_ylabel("radio (km)")
axes[1].legend(ncol=2, fontsize=8)
fig.autofmt_xdate()
fig.savefig(FIG_DIR / "radii_timeseries.png", dpi=150, bbox_inches="tight")

## 3. Perfiles radiales en el tiempo de verificación: Rankine (`wind.py`) frente a W06

In [ ]:
r = np.linspace(1.0, 1200.0, 2400)
vmax, lat, rmw = float(row["vmax_kt"]), float(row["lat"]), float(row["rmw_km"])
rmw_used = rmw if np.isfinite(rmw) else float(row["obs_rmw_rmax_km"])

v_rankine = compute_radial_wind_profile(r, vmax_kt=vmax, rmw_km=rmw_used)
profiles = {
    "Rankine (wind.py)": v_rankine,
    "W06 pa21": Willoughby2006Profile(vmax * KNOT_TO_MS, lat, rmax_variant="pa21")(r) / KNOT_TO_MS,
    "W06 obs_rmw": Willoughby2006Profile(vmax * KNOT_TO_MS, lat, rmw_used)(r) / KNOT_TO_MS,
    "W06 calib_r34": Willoughby2006Profile(
        vmax * KNOT_TO_MS, lat, rmw_used, x1_km=float(row["calib_r34_x1_km"])
    )(r) / KNOT_TO_MS,
}

fig, ax = plt.subplots(figsize=(8, 4.5))
for label, v in profiles.items():
    ax.plot(r, v, label=label)
ax.axhline(34, color="0.4", ls="--", lw=0.8)
ax.axhline(2.0 / KNOT_TO_MS, color="0.6", ls=":", lw=0.8)
if np.isfinite(row["r34_obs_km"]):
    ax.axvline(row["r34_obs_km"], color="k", lw=0.8, label="R34 obs")
ax.set(xlabel="r (km)", ylabel="V (kt)", title=f"{STORM_NAME} {T_CHECK:%Y-%m-%d %H} UTC")
ax.legend(fontsize=8)
fig.savefig(FIG_DIR / "radial_profiles.png", dpi=150, bbox_inches="tight")

# Discontinuidad de V* en R34 con Rankine: V(R34) debería ser 34 kt
if np.isfinite(row["r34_obs_km"]):
    v_at_r34 = float(compute_radial_wind_profile(np.array([row["r34_obs_km"]]), vmax, rmw_used)[0])
    print(f"Rankine: V(R34 obs) = {v_at_r34:.1f} kt -> V* en el borde = "
          f"{max(v_at_r34 - 34, 0) / (vmax - 34):.2f}")

## 4. MSWEP diario

Primero comprueba nomenclatura y convención temporal. Con `MSWEP_DAILY_STAMP = "start"`, el archivo con marca D contiene la acumulación de 00 a 24 UTC del día D. Si al comparar con los archivos de 3 h resulta ser el fin de la acumulación, cambia el parámetro a `"end"`.

In [ ]:
def mswep_daily_path(day: pd.Timestamp) -> Path:
    return MSWEP_DAILY_DIR / MSWEP_FILE_PATTERN.format(date=day)


days = pd.date_range(track["time"].min().floor("D"), track["time"].max().floor("D"), freq="D")
bbox = dict(
    lon_min=float(track["lon"].min()) - PAD_DEG, lon_max=float(track["lon"].max()) + PAD_DEG,
    lat_min=max(float(track["lat"].min()) - PAD_DEG, -60.0),
    lat_max=min(float(track["lat"].max()) + PAD_DEG, 60.0),
)
missing = [d for d in days if not mswep_daily_path(d).exists()]
print(f"{len(days)} días; faltan {len(missing)} archivos", missing[:3])

probe = read_mswep_dataset(mswep_daily_path(days[0]))
print(probe)
print(probe[PRECIP_VAR].attrs)

In [ ]:
def read_daily_precipitation(day: pd.Timestamp) -> xr.DataArray:
    # Campo diario asignado al día calendario 'day' (selección por día, no por cercanía)
    file_day = day if MSWEP_DAILY_STAMP == "start" else day + pd.Timedelta(days=1)
    data = read_mswep_dataset(mswep_daily_path(file_day))
    precip = standardize_mswep_time(extract_mswep_precipitation(data, PRECIP_VAR))
    precip = subset_mswep_domain(precip, **bbox)
    return precip.isel(valid_time=0, drop=True).load()


P0 = read_daily_precipitation(days[0])
LAT_NAME, LON_NAME = infer_mswep_lat_lon_names(P0)
LON2D, LAT2D = np.meshgrid(P0[LON_NAME].values, P0[LAT_NAME].values)
print(P0.shape, float(P0.max()), P0.attrs.get("units"))

## 5. Climatología diaria Q90/Q95/Q99

Se calcula sobre la misma caja espacial y se guarda para reutilizarla. Abrir ~5 000 archivos diarios tarda varios minutos; la segunda vez se lee del archivo. Los percentiles incluyen días secos, igual que el análisis de P95 del proyecto; si decides usar solo días húmedos, cambia aquí y documenta la decisión.

In [ ]:
if CLIM_FILE.exists():
    clim = xr.open_dataset(CLIM_FILE).load()
else:
    clim_days = pd.date_range(f"{CLIM_YEARS[0]}-01-01", f"{CLIM_YEARS[1]}-12-31", freq="D")
    clim_files = [
        mswep_daily_path(d) for d in clim_days
        if d.month in CLIM_MONTHS and mswep_daily_path(d).exists()
    ]
    print(f"{len(clim_files)} archivos para la climatología")

    ds_all = xr.open_mfdataset(
        clim_files, combine="nested", concat_dim="time",
        preprocess=lambda d: subset_mswep_domain(d[[PRECIP_VAR]], **bbox),
        chunks={}, parallel=False,
    )
    p_all = ds_all[PRECIP_VAR].chunk({"time": -1, LAT_NAME: "auto", LON_NAME: "auto"})
    clim = xr.Dataset(compute_standard_precipitation_climatology(p_all, dim="time")).compute()
    CLIM_FILE.parent.mkdir(parents=True, exist_ok=True)
    clim.to_netcdf(CLIM_FILE)

Q90, Q95, Q99 = (np.asarray(q.values, dtype=float) for q in extract_standard_percentiles(clim))
assert Q90.shape == LON2D.shape, "la climatología y MSWEP deben compartir malla"
print({k: float(np.nanmedian(v.values)) for k, v in clim.data_vars.items()})

## 6. ITCHI diario (opción B)

Para cada día: se evalúa el pipeline en cada tiempo sinóptico con el campo diario y se toma el máximo intradiario. Radio de atribución: radio exterior W06 simétrico (en lugar de ROCLOUD). Radio directo: R34 observado; W06 R34 como respaldo.

In [ ]:
def run_snapshot(row: pd.Series, precip2d: np.ndarray, prefix: str, wind_profile: str) -> dict:
    rmw = float(row["rmw_km"]) if np.isfinite(row["rmw_km"]) else float(row[f"{prefix}rmax_km"])
    r_outer = float(row[f"{prefix}r_outer_km"])
    wind = None
    if wind_profile == "w06":
        radius = build_geometry_fields(LON2D, LAT2D, row["lon"], row["lat"])["radius_km"]
        wind = compute_w06_wind_hazard(
            radius, row["vmax_kt"], row["lat"], rmw_km=rmw, x1_km=float(row[f"{prefix}x1_km"])
        )
    return compute_itchi_snapshot_from_grid(
        precipitation=precip2d, q90=Q90, q95=Q95, q99=Q99,
        lon=LON2D, lat=LAT2D, center_lon=float(row["lon"]), center_lat=float(row["lat"]),
        r34_by_quadrant={q: row[DEFAULT_R34_COLUMN_MAP[q]] for q in QUADRANTS}, r34_unit="nm",
        rocloud_by_quadrant={q: r_outer for q in QUADRANTS}, rocloud_unit="km",
        wind_hazard_normalized=wind, vmax_kt=float(row["vmax_kt"]), rmw_km=rmw,
        fallback_direct_radius_km=parametric_direct_fallback_km(row, f"{prefix}r34_km"),
        run_quality_control=True,
    )


FIELDS = ("ITCHI", "H_dir", "H_ind", "H_W", "H_P")


def run_event(prefix: str = "calib_r34_", wind_profile: str = "rankine") -> xr.Dataset:
    daily, kept_days, n_snap = [], [], []
    for day in days:
        rows = track[track["time"].dt.floor("D") == day]
        rows = rows[np.isfinite(rows[f"{prefix}r_outer_km"])]
        if rows.empty:
            continue
        precip2d = np.asarray(read_daily_precipitation(day).values, dtype=float)
        snaps = [run_snapshot(r, precip2d, prefix, wind_profile) for _, r in rows.iterrows()]
        out = {k: np.nanmax(np.stack([s[k] for s in snaps]), axis=0) for k in FIELDS}
        out["M_direct"] = np.any(np.stack([s["M_direct"] for s in snaps]), axis=0)
        out["M_indirect"] = np.any(np.stack([s["M_indirect"] for s in snaps]), axis=0)
        daily.append(out); kept_days.append(day); n_snap.append(len(snaps))
    coords = {"day": kept_days, LAT_NAME: P0[LAT_NAME].values, LON_NAME: P0[LON_NAME].values}
    dims = ("day", LAT_NAME, LON_NAME)
    ds = xr.Dataset({k: (dims, np.stack([d[k] for d in daily])) for k in daily[0]}, coords=coords)
    ds["n_snapshots"] = ("day", n_snap)
    ds["ITCHI_max"] = compute_event_max(ds["ITCHI"], dim="day")
    ds["ITCHI_acc"] = compute_event_accumulated(ds["ITCHI"], dim="day")
    ds.attrs.update(storm=f"{STORM_NAME} {SEASON}", sid=SID, radii=prefix.rstrip("_"),
                    wind_profile=wind_profile, precipitation="MSWEP daily, option B")
    return ds


event = run_event("calib_r34_", wind_profile=PR.get("wind_profile", "rankine"))
event

## 7. Mapas

In [ ]:
def map_axes(n: int, figsize=(15, 4.5)):
    fig, axes = plt.subplots(1, n, figsize=figsize, subplot_kw={"projection": ccrs.PlateCarree()})
    for ax in np.atleast_1d(axes):
        ax.coastlines(lw=0.6)
        ax.plot(track["lon"], track["lat"], "k.-", lw=0.8, ms=3, transform=ccrs.PlateCarree())
    return fig, np.atleast_1d(axes)


day_check = T_CHECK.floor("D")
snap = event.sel(day=day_check)
fig, axes = map_axes(3)
for ax, name in zip(axes, ("H_dir", "H_ind", "ITCHI")):
    plot_itchi_field(snap[name].values, ax, lon=LON2D, lat=LAT2D, crs=ccrs.PlateCarree(),
                     colorbar_label=name)
    ax.set_title(f"{name} {day_check:%Y-%m-%d}")
fig.savefig(FIG_DIR / "itchi_day_components.png", dpi=150, bbox_inches="tight")

fig, axes = map_axes(2, figsize=(11, 4.5))
for ax, name in zip(axes, ("ITCHI_max", "ITCHI_acc")):
    plot_itchi_field(event[name].values, ax, lon=LON2D, lat=LAT2D, crs=ccrs.PlateCarree(),
                     colorbar_label=name)
    ax.set_title(f"{name} — {STORM_NAME} {SEASON}")
fig.savefig(FIG_DIR / "itchi_event.png", dpi=150, bbox_inches="tight")

## 8. Sensibilidad

- Perfil de viento: Rankine (`wind.py`) frente a W06 calibrado a R34.
- Umbral exterior: 2 m s⁻¹ (PA21) frente a 5 kt (Knaff et al. 2014).

In [ ]:
event_w06 = run_event("calib_r34_", wind_profile="w06")

track = add_parametric_radii(
    track, use_observed_rmw=True, calibrate_to_r34=True, rmw_unit=RMW_UNIT, r34_unit="nm",
    thresholds_ms={"r_outer_km": V_R5_MS, "r34_km": V_R34_MS}, prefix="calib_r5_",
)
event_r5 = run_event("calib_r5_", wind_profile=event.attrs["wind_profile"])

diffs = {
    "ΔITCHI_max viento (W06 − Rankine)": event_w06["ITCHI_max"] - event["ITCHI_max"],
    "ΔITCHI_max umbral (5 kt − 2 m/s)": event_r5["ITCHI_max"] - event["ITCHI_max"],
}
fig, axes = map_axes(2, figsize=(11, 4.5))
for ax, (label, d) in zip(axes, diffs.items()):
    m = ax.pcolormesh(LON2D, LAT2D, d.values, cmap="RdBu_r", vmin=-0.5, vmax=0.5,
                      transform=ccrs.PlateCarree())
    fig.colorbar(m, ax=ax, shrink=0.8); ax.set_title(label, fontsize=10)
    print(f"{label}: media = {float(d.mean()):+.4f}, |Δ|>0.1 en {float((abs(d) > 0.1).mean()) * 100:.2f} % de celdas")
fig.savefig(FIG_DIR / "itchi_sensitivity.png", dpi=150, bbox_inches="tight")

print("Δr_out medio (5 kt − 2 m/s):",
      float((track["calib_r5_r_outer_km"] - track["calib_r34_r_outer_km"]).mean()), "km")

## 9. Resumen diario y guardado

In [ ]:
summary = pd.DataFrame({
    "n_snapshots": event["n_snapshots"].values,
    "ITCHI_max_dia": event["ITCHI"].max(dim=(LAT_NAME, LON_NAME)).values,
    "area_ITCHI>0.5 (celdas)": (event["ITCHI"] > 0.5).sum(dim=(LAT_NAME, LON_NAME)).values,
    "celdas_directas": event["M_direct"].sum(dim=(LAT_NAME, LON_NAME)).values,
    "celdas_indirectas": event["M_indirect"].sum(dim=(LAT_NAME, LON_NAME)).values,
}, index=pd.DatetimeIndex(event["day"].values, name="día"))
display(summary)

out_dir = ROOT / CFG["outputs"]["event_products"]
out_dir.mkdir(parents=True, exist_ok=True)
event.to_netcdf(out_dir / f"itchi_{STORM_NAME.lower()}{SEASON}_daily_optionB.nc")